<a href="https://www.kaggle.com/code/ammarnael/lstm-cnn?scriptVersionId=308627778" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

In [2]:

df = pd.read_csv('/kaggle/input/datasets/varpit94/apple-stock-data-updated-till-22jun2021/AAPL.csv')

df.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,1980-12-12,0.128348,0.128906,0.128348,0.128348,0.100323,469033600
1,1980-12-15,0.122210,0.122210,0.121652,0.121652,0.095089,175884800
2,1980-12-16,0.113281,0.113281,0.112723,0.112723,0.088110,105728000
3,1980-12-17,0.115513,0.116071,0.115513,0.115513,0.090291,86441600
4,1980-12-18,0.118862,0.119420,0.118862,0.118862,0.092908,73449600


In [3]:
df = df.sort_values("Date")

df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

df.head()

,Open,High,Low,Close,Volume
0,0.128348,0.128906,0.128348,0.128348,469033600
1,0.122210,0.122210,0.121652,0.121652,175884800
2,0.113281,0.113281,0.112723,0.112723,105728000
3,0.115513,0.116071,0.115513,0.115513,86441600
4,0.118862,0.119420,0.118862,0.118862,73449600


In [4]:
scaler = MinMaxScaler()

data = scaler.fit_transform(df)

data[:5]

array([[0.00043095, 0.00043327, 0.00044251, 0.00043548, 0.0631981 ],
       [0.00039733, 0.00039666, 0.00040512, 0.00039868, 0.02369891],
       [0.00034843, 0.00034784, 0.00035526, 0.00034961, 0.01424591],
       [0.00036065, 0.00036309, 0.00037084, 0.00036495, 0.01164724],
       [0.00037899, 0.0003814 , 0.00038954, 0.00038335, 0.00989668]])

In [5]:
def create_data(data, seq_len=30):
    X = []
    y = []

    for i in range(len(data) - seq_len - 1):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len][3])  # Close price

    return np.array(X), np.array(y)

seq_len = 30

X, y = create_data(data, seq_len)

X.shape, y.shape

((10378, 30, 5), (10378,))

In [6]:
split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [7]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_data = StockDataset(X_train, y_train)
test_data = StockDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

In [8]:
class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(5, 32, 3)
        self.conv2 = nn.Conv1d(32, 64, 3)

        self.relu = nn.ReLU()

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=50,
            batch_first=True
        )

        self.fc = nn.Linear(50, 1)

    def forward(self, x):

        x = x.permute(0, 2, 1)

        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))

        x = x.permute(0, 2, 1)

        out, _ = self.lstm(x)

        out = self.fc(out[:, -1, :])

        return out.squeeze()

In [9]:
model = CNN_LSTM()

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
epochs = 10

for epoch in range(epochs):

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        preds = model(X_batch)

        loss = loss_fn(preds, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss)

Epoch: 1 Loss: 0.05233408810886431
Epoch: 2 Loss: 0.0004891115299727744
Epoch: 3 Loss: 0.00044861079715019514
Epoch: 4 Loss: 0.0004084822822107981
Epoch: 5 Loss: 0.00039882284882253316
Epoch: 6 Loss: 0.00045370572817660104
Epoch: 7 Loss: 0.00047463915246481747
Epoch: 8 Loss: 0.0004673945383117939
Epoch: 9 Loss: 0.0004690368836577363
Epoch: 10 Loss: 0.0004135570153351864


In [11]:
model.eval()

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

preds = model(X_test_tensor).detach().numpy()

preds[:10]

array([0.10531414, 0.10595822, 0.10701802, 0.10763692, 0.10781679,
       0.10760661, 0.10729682, 0.10660462, 0.10577254, 0.10513328],
      dtype=float32)

In [12]:
print("Predicted:", preds[:10])
print("Actual:", y_test[:10])

Predicted: [0.10531414 0.10595822 0.10701802 0.10763692 0.10781679 0.10760661
 0.10729682 0.10660462 0.10577254 0.10513328]
Actual: [0.11114929 0.11040933 0.10966152 0.10856827 0.10984405 0.10829544
 0.10591072 0.10648972 0.10572622 0.10639747]
